# Data Mining Assignment — Data Lake Preprocessing with Apache Spark

**Name:** Mohd Haider  
**Roll Number:** LCS2023038

This notebook creates a simple Bronze/Silver Data Lake structure, inspects the raw `sales.csv` using PySpark, performs basic preprocessing, saves the Silver output, and verifies the result.

> The Bronze copy is kept unchanged. No Gold layer or detailed analysis is performed.

This notebook is an independent implementation of the assignment requirements; the supplied raw dataset is intentionally not modified.


In [ ]:
# Step 1 — Install the Spark dependency
!pip -q install pyspark


In [ ]:
# Step 2 — Imports and data-lake locations
import os
import shutil
from pyspark.sql import SparkSession, functions as F, types as T

LAKE_ROOT = "/content/data_lake"
BRONZE_DIR = f"{LAKE_ROOT}/bronze"
SILVER_DIR = f"{LAKE_ROOT}/silver_df/processed_sales"

os.makedirs(BRONZE_DIR, exist_ok=True)
os.makedirs(SILVER_DIR, exist_ok=True)

print("Bronze layer:", BRONZE_DIR)
print("Silver layer:", SILVER_DIR)


## 3. Load the provided dataset into the Bronze layer

The original dataset is copied without editing. If `sales.csv` is already uploaded to Colab, it is used directly. Otherwise, the cell downloads the class-provided dataset from the assignment repository.


In [ ]:
# Copy the untouched source file into the Bronze layer

source_file = "/content/sales.csv"
bronze_path = f"{BRONZE_DIR}/sales.csv"

if not os.path.exists(source_file):
    !wget -q -O /content/sales.csv https://raw.githubusercontent.com/necromancersanju-netizen/data-mining-data-lake-assignment/main/dataset/sales.csv

shutil.copy2(source_file, bronze_path)
print("Bronze file created:", bronze_path)


In [ ]:
# Start a local Spark session
spark = (
    SparkSession.builder
    .appName("DataMiningDataLakeAssignment")
    .master("local[*]")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print("Spark version:", spark.version)


## 5. Read the Bronze layer and inspect the source

In [ ]:
df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(bronze_path)
)

print("Schema:")
df.printSchema()

print("First 5 rows:")
df.show(5, truncate=False)

print("Total records:", df.count())


## 6. Initial data-quality checks

The checks below identify:
- exact duplicate rows
- null values
- text-format inconsistencies
- invalid quantity
- invalid unit price
- invalid discount percentage
- invalid dates


In [ ]:
# Exact duplicate rows
duplicate_count = df.count() - df.dropDuplicates().count()
print("Exact duplicate rows:", duplicate_count)

# Null counts by column
null_counts = df.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in df.columns
])
print("Null counts:")
null_counts.show()


In [ ]:
# Text-format inconsistencies: distinct category values after trimming/lowercasing
for c in ["category", "payment_method", "city", "state", "order_status"]:
    if c in df.columns:
        print(f"\n{c}:")
        df.select(c).where(F.col(c).isNotNull()).distinct().orderBy(c).show(100, truncate=False)


In [ ]:
# Invalid numeric values
numeric_checks = df.select(
    F.sum((F.col("quantity").isNull() | (F.col("quantity") <= 0)).cast("int")).alias("invalid_quantity"),
    F.sum((F.col("unit_price").isNull() | (F.col("unit_price") <= 0)).cast("int")).alias("invalid_unit_price"),
    F.sum((F.col("discount_percent").isNull() | (F.col("discount_percent") < 0) | (F.col("discount_percent") > 100)).cast("int")).alias("invalid_discount")
)
numeric_checks.show()


In [ ]:
# Safely interpret the supported date formats.
def parse_order_date(col_name):
    value = F.trim(F.col(col_name).cast("string"))
    return F.coalesce(
        F.try_to_timestamp(value, F.lit("yyyy-MM-dd")),
        F.try_to_timestamp(value, F.lit("dd/MM/yyyy")),
        F.try_to_timestamp(value, F.lit("MM/dd/yyyy"))
    ).cast("date")

date_check = df.withColumn("parsed_order_date", parse_order_date("order_date"))

print("Rows with invalid/unparseable dates:")
date_check.filter(
    F.col("order_date").isNotNull() & F.col("parsed_order_date").isNull()
).select("order_id", "order_date").show(50, truncate=False)


## 7. Build the cleaned Silver dataset

Rules used:
1. Remove exact duplicate rows.
2. Trim whitespace in text fields.
3. Standardize obvious capitalization inconsistencies in text fields.
4. Convert numeric columns to numeric types.
5. Treat non-positive quantity and unit price as invalid and replace them with null.
6. Treat discount values outside 0–100% as invalid and replace them with null.
7. Parse valid dates from `yyyy-MM-dd`, `dd/MM/yyyy`, or `MM/dd/yyyy` into one consistent date type; clearly invalid dates become null.
8. Fill missing text values with `"Unknown"` and numeric missing values with the median of the valid Silver records.
9. Keep useful records rather than dropping rows unnecessarily.

In [ ]:
# Remove duplicate records and normalize text spacing
silver_df = df.dropDuplicates()

text_cols = ["customer_name", "product", "category", "payment_method", "city", "state", "order_status"]
for c in text_cols:
    if c in silver_df.columns:
        silver_df = silver_df.withColumn(c, F.trim(F.col(c)))

# Standardize obvious capitalization inconsistencies.
silver_df = silver_df.withColumn("category", F.initcap(F.lower(F.col("category"))))
silver_df = silver_df.withColumn("payment_method", F.initcap(F.lower(F.col("payment_method"))))
silver_df = silver_df.withColumn("city", F.initcap(F.lower(F.col("city"))))
silver_df = silver_df.withColumn("state", F.initcap(F.lower(F.col("state"))))
silver_df = silver_df.withColumn("order_status", F.initcap(F.lower(F.col("order_status"))))


In [ ]:
# Validate the numeric fields
silver_df = (
    silver_df
    .withColumn("quantity", F.col("quantity").cast("int"))
    .withColumn("unit_price", F.col("unit_price").cast("double"))
    .withColumn("discount_percent", F.col("discount_percent").cast("double"))
)

silver_df = silver_df.withColumn(
    "quantity",
    F.when(F.col("quantity") > 0, F.col("quantity")).otherwise(F.lit(None).cast("int"))
)

silver_df = silver_df.withColumn(
    "unit_price",
    F.when(F.col("unit_price") > 0, F.col("unit_price")).otherwise(F.lit(None).cast("double"))
)

silver_df = silver_df.withColumn(
    "discount_percent",
    F.when(
        (F.col("discount_percent") >= 0) & (F.col("discount_percent") <= 100),
        F.col("discount_percent")
    ).otherwise(F.lit(None).cast("double"))
)


In [ ]:
# Convert the cleaned date text to a Spark date.
# Invalid calendar dates become NULL instead of raising a parsing exception.
silver_df = silver_df.withColumn(
    "order_date",
    parse_order_date("order_date")
)


In [ ]:
# Complete the missing-value treatment
# Text: explicit Unknown label.
for c in ["customer_name", "product", "category", "payment_method", "city", "state", "order_status"]:
    silver_df = silver_df.withColumn(c, F.coalesce(F.col(c), F.lit("Unknown")))

# Numeric: median of valid values, calculated with Spark approxQuantile.
for c in ["quantity", "unit_price", "discount_percent"]:
    median = silver_df.approxQuantile(c, [0.5], 0.01)[0]
    silver_df = silver_df.withColumn(c, F.coalesce(F.col(c), F.lit(float(median))))

# Invalid dates are rare/clearly invalid, so retain the record but label the date as null.


In [ ]:
# Write the Silver result as one CSV file
# Spark writes CSV output as a directory, so coalesce(1) creates one part file.
silver_output_dir = f"{SILVER_DIR}/csv_output"

(
    silver_df.coalesce(1)
    .write
    .mode("overwrite")
    .option("header", True)
    .csv(silver_output_dir)
)

# Rename the single part file to the required sales_silver.csv
part_files = [
    f for f in os.listdir(silver_output_dir)
    if f.startswith("part-") and f.endswith(".csv")
]
assert len(part_files) == 1, f"Expected one CSV part, found {part_files}"

final_csv = f"{SILVER_DIR}/sales_silver.csv"
if os.path.exists(final_csv):
    os.remove(final_csv)

os.rename(
    os.path.join(silver_output_dir, part_files[0]),
    final_csv
)
shutil.rmtree(silver_output_dir)

print("Silver CSV saved to:", final_csv)
print("Silver CSV size (bytes):", os.path.getsize(final_csv))

## 8. Verify the processed output

In [ ]:
before_count = df.count()
after_count = silver_df.count()

print("Row count before preprocessing:", before_count)
print("Row count after preprocessing:", after_count)

print("\nSilver schema:")
silver_df.printSchema()

print("\nSample Silver rows:")
silver_df.show(10, truncate=False)


In [ ]:
# Final checks
print("Remaining exact duplicates:", after_count - silver_df.dropDuplicates().count())

print("Remaining null counts:")
silver_df.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in silver_df.columns
]).show()

print("Silver output exists:", os.path.exists(final_csv))
if os.path.exists(final_csv):
    print("Silver file path:", final_csv)

## 9. Preprocessing summary

- Exact duplicate rows were removed.
- Leading/trailing spaces were trimmed from text fields.
- Obvious text capitalization inconsistencies were standardized.
- Quantity and unit price values that were non-positive were treated as invalid.
- Discount percentages outside 0–100 were treated as invalid.
- Valid dates were converted to a consistent date type; clearly invalid dates were retained as null rather than deleting the whole record.
- Missing text values were represented as `Unknown`.
- Missing/invalid numeric values were filled using median values calculated from valid records.
- The processed result was saved as `sales_silver.csv` in the Silver layer.
- No Gold layer or detailed analysis was performed.
